# Silver layer

The silver layer is build up like this:  
1.  import bronze df and check.  
2.  quick sanity checks, col present, missing val, data type.  
3.  Define doc boundary and silver schema.  
4.  one coherent text (silver_text).  
5.  inspect silver_text.  
6.  what cleaning is allowed?  
7.  start preprocessing.  
8.  spaCY annotation.  
9.  output to gold.  

In [74]:
# Import needed libraries
import pandas as pd
import spacy
import os
import re

# Import bronze dataframe + sanity checks

In [75]:
# Load bronze data
project_data = pd.read_csv("../Data/bronze/bronze_data.csv")

# check the required columns
required_columns = ['id', 'content', 'title']
missing_columns = list(set(required_columns) - set(project_data.columns))

if missing_columns:
    print(f"Error: Missing columns in the dataset: {missing_columns}")
else:
    print(f"All required columns are present: {list(project_data.columns)}")

# Print project_data info
print(project_data.info())

# Check for missing values in the required columns
missing_values = project_data[required_columns].isnull().sum()
print(f"Missing values in required columns:\n{missing_values}")



All required columns are present: ['id', 'title', 'content']
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       1 non-null      str  
 1   title    1 non-null      str  
 2   content  1 non-null      str  
dtypes: str(3)
memory usage: 156.0 bytes
None
Missing values in required columns:
id         0
content    0
title      0
dtype: int64


Now we imported the data correctly, we go on to the pre-NLP preprocessing steps.  
We make the data ready for NLP usage, this mean we will check for:  
- concatenate to text.    
- Normalize the whitespace.  
- Split paragraphs.  
- dedupe prarapraphs.  

In [76]:
# First we will concatenate the title and content columns to create a single text column for processing
def concat_to_text(x):
    if x is None:
        return ""
    # x is expected to be [title, content]
    parts = [str(part).strip() for part in x if part is not None and str(part).strip()]
    return "\n\n".join(parts)

# Concatenate title and content into a new text column
project_data['text'] = project_data.apply(lambda row: concat_to_text([row['title'], row['content']]), axis=1)
print(project_data['text'])

print(project_data.head())

0    Assessing Climate Adaptation Measures in Low-L...
Name: text, dtype: str
       id                                              title  \
0  doc_01  Assessing Climate Adaptation Measures in Low-L...   

                                             content  \
0  Introduction\nClimate adaptation has become a ...   

                                                text  
0  Assessing Climate Adaptation Measures in Low-L...  


In [77]:
# Now we will normalize the whitespace in the text column
def normalize_whitespace(text):
    if text is None:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)      # collapse spaces
    text = re.sub(r"\n{3,}", "\n\n", text)   # keep paragraph breaks
    return text.strip()
project_data['text'] = project_data['text'].apply(normalize_whitespace)
print(project_data['text'])

0    Assessing Climate Adaptation Measures in Low-L...
Name: text, dtype: str


In [78]:
# now we can create the silver_text column which will be used for the silver layer
project_data['silver_text'] = project_data['text']
print(project_data['silver_text'])

print(project_data.head())


0    Assessing Climate Adaptation Measures in Low-L...
Name: silver_text, dtype: str
       id                                              title  \
0  doc_01  Assessing Climate Adaptation Measures in Low-L...   

                                             content  \
0  Introduction\nClimate adaptation has become a ...   

                                                text  \
0  Assessing Climate Adaptation Measures in Low-L...   

                                         silver_text  
0  Assessing Climate Adaptation Measures in Low-L...  


In [79]:
# Select only Silver-relevant columns
silver_df = project_data[["id", "silver_text"]].copy()

# Reset index for clean export
silver_df = silver_df.reset_index(drop=True)

print(silver_df.head())

       id                                        silver_text
0  doc_01  Assessing Climate Adaptation Measures in Low-L...


In [80]:
silver_path = "../Data/silver/silver_data.csv"

silver_df.to_csv(silver_path, index=False, encoding="utf-8")

print(f"Silver data saved to: {silver_path}")

Silver data saved to: ../Data/silver/silver_data.csv
